# Three-role LLM smoke test

Verifies that the RAG generator, synth-GT author, and judge roles are reachable
through a single `ChatCompletionsClient` surface, and that the three-distinct-models
invariant holds.

**Prereqs:** `.env` is populated. `FOUNDRY_SYNTH_GT_*` and `FOUNDRY_JUDGE_*` must
point to non-OpenAI-family catalog models click-deployed into the Foundry project.

In [1]:
from sentcite.llm import describe_roles, get_client, get_model_id

# Fails loudly if .env is missing a role or two roles share a family.
roles = describe_roles()
roles

{'rag': 'gpt-4.1-1',
 'synth_gt': 'mistral-large-3',
 'judge': 'llama-3.3-70b-instruct'}

In [2]:
from azure.ai.inference.models import SystemMessage, UserMessage

PROMPT = [
    SystemMessage(content="Reply with exactly the word: OK"),
    UserMessage(content="ping"),
]

for role in ("rag", "synth_gt", "judge"):
    client = get_client(role)
    resp = client.complete(model=get_model_id(role), messages=PROMPT, max_tokens=5)
    print(f"{role:10s} {roles[role]:45s} -> {resp.choices[0].message.content!r}")

rag        gpt-4.1-1                                     -> 'OK'


synth_gt   mistral-large-3                               -> 'OK'


judge      llama-3.3-70b-instruct                        -> 'OK'
